# 17 — Plan A: Germany Interactive Web Map

Notebook 17 is the visualization/delivery layer of Plan A.

## Purpose

Create one Germany-wide interactive map for the **52,214 existing bridges** using:

- canonical bridge-map data from Notebook 00;
- current-condition predictions from Notebook 15;
- future-condition scenarios from Notebook 16.

### Supported views

- Current predicted condition
- Observed condition
- +10 years
- +25 years
- +50 years

### Filters

- Bundesland
- Bauwerksart
- Baustoff
- condition range
- bridge ID search

### Important architecture boundary

Notebook 17:

- does not retrain any ML model;
- does not forecast again;
- does not perform FEM or structural design;
- does not modify the frozen model;
- does not reconstruct future values from an annual deterioration rate.

All future values are consumed directly from Notebook 16.

The map is an interface for Plan A, not a structural-design engine.

## 00 — Input / output contract

### Inputs

```text
Dataset_PlanA-B/
└── Map/
    └── bridges_map.parquet

Output_PlanA-B/
├── 15_Plan_A_Current_Bridge_Condition/
│   └── plan_a_current_condition.parquet
└── 16_Plan_A_Future_Condition/
    └── plan_a_future_condition.parquet
```

### Outputs

```text
Output_PlanA-B/
└── 17_Plan_A_Germany_Web_Map/
    ├── plan_a_germany_bridge_map.html
    ├── plan_a_map_data.parquet
    ├── plan_a_map_data.csv
    ├── 17_plan_a_map_summary.csv
    └── 17_plan_a_map_manifest.json
```

The HTML map is standalone except for standard Leaflet CDN assets loaded by the browser.

In [1]:
from pathlib import Path
import json
import hashlib
import html
import os
import sys

import numpy as np
import pandas as pd

try:
    from pyproj import Transformer
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "pyproj is required for the controlled EPSG:3857 -> EPSG:4326 "
        "bridge-coordinate transformation."
    ) from exc

PROJECT_ROOT = Path(r"C:\Datenanalyse\final Project")
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"

MAP_INPUT = DATASET_ROOT / "Map" / "bridges_map.parquet"
CURRENT_INPUT = (
    OUTPUT_ROOT
    / "15_Plan_A_Current_Bridge_Condition"
    / "plan_a_current_condition.parquet"
)
FUTURE_INPUT = (
    OUTPUT_ROOT
    / "16_Plan_A_Future_Condition"
    / "plan_a_future_condition.parquet"
)

OUTPUT_DIR = OUTPUT_ROOT / "17_Plan_A_Germany_Web_Map"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HTML_OUTPUT = OUTPUT_DIR / "plan_a_germany_bridge_map.html"
MAP_PARQUET = OUTPUT_DIR / "plan_a_map_data.parquet"
MAP_CSV = OUTPUT_DIR / "plan_a_map_data.csv"
SUMMARY_CSV = OUTPUT_DIR / "17_plan_a_map_summary.csv"
MANIFEST_JSON = OUTPUT_DIR / "17_plan_a_map_manifest.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MAP_INPUT:", MAP_INPUT)
print("CURRENT_INPUT:", CURRENT_INPUT)
print("FUTURE_INPUT:", FUTURE_INPUT)
print("OUTPUT_DIR:", OUTPUT_DIR)

for p in [MAP_INPUT, CURRENT_INPUT, FUTURE_INPUT]:
    if not p.exists():
        raise FileNotFoundError(f"Required input not found:\n{p}")

print("[PASS] all Notebook 17 inputs exist")

PROJECT_ROOT: C:\Datenanalyse\final Project
MAP_INPUT: C:\Datenanalyse\final Project\Dataset_PlanA-B\Map\bridges_map.parquet
CURRENT_INPUT: C:\Datenanalyse\final Project\Output_PlanA-B\15_Plan_A_Current_Bridge_Condition\plan_a_current_condition.parquet
FUTURE_INPUT: C:\Datenanalyse\final Project\Output_PlanA-B\16_Plan_A_Future_Condition\plan_a_future_condition.parquet
OUTPUT_DIR: C:\Datenanalyse\final Project\Output_PlanA-B\17_Plan_A_Germany_Web_Map
[PASS] all Notebook 17 inputs exist


## 01 — Load and validate the three canonical inputs

The three layers have different roles:

```text
Notebook 00 map dataset
    = geometry + bridge metadata + traffic context

Notebook 15
    = current condition prediction

Notebook 16
    = future condition scenarios
```

They are joined only through `bridge_id`.

In [2]:
bridge_map = pd.read_parquet(MAP_INPUT)
current_df = pd.read_parquet(CURRENT_INPUT)
future_df = pd.read_parquet(FUTURE_INPUT)

print("bridge_map:", bridge_map.shape)
print("current_df:", current_df.shape)
print("future_df:", future_df.shape)

for name, df in [
    ("bridge_map", bridge_map),
    ("current_df", current_df),
    ("future_df", future_df),
]:
    if "bridge_id" not in df.columns:
        raise KeyError(f"{name} does not contain bridge_id.")

    df["bridge_id"] = df["bridge_id"].astype("string").str.strip()

    if df["bridge_id"].isna().any():
        raise ValueError(f"{name} contains missing bridge_id values.")

    if not df["bridge_id"].is_unique:
        raise ValueError(f"{name} contains duplicate bridge_id values.")

if len(bridge_map) != 52214:
    raise ValueError(
        f"Expected 52,214 bridge-map rows; found {len(bridge_map)}"
    )

if len(current_df) != 52214:
    raise ValueError(
        f"Expected 52,214 current-condition rows; found {len(current_df)}"
    )

if len(future_df) != 52214:
    raise ValueError(
        f"Expected 52,214 future-condition rows; found {len(future_df)}"
    )

print("[PASS] input populations validated")

bridge_map: (52214, 17)
current_df: (52214, 17)
future_df: (52214, 23)
[PASS] input populations validated


## 02 — Validate the bridge-map schema

The existing `geom_x` / `geom_y` values are source GIS coordinates, not WGS84 longitude/latitude.

The project geospatial lineage establishes the controlled map transformation:

```text
EPSG:3857
   ↓
EPSG:4326
```

No CRS is guessed from numeric ranges.

In [3]:
required_map_columns = [
    "bridge_id",
    "bauwerksart_text",
    "baujahr",
    "laenge",
    "breite",
    "baustoffklasse",
    "zustandsnote",
    "zustandsnotenklasse",
    "gis_ort",
    "gis_kreis",
    "gis_bundesland",
    "geom_x",
    "geom_y",
    "traffic_dtv_latest",
    "traffic_dtv_mean",
]

missing_map = [
    c for c in required_map_columns
    if c not in bridge_map.columns
]

if missing_map:
    raise KeyError(
        f"Bridge-map dataset is missing required columns: {missing_map}"
    )

for c in [
    "bauwerksart_text",
    "baustoffklasse",
    "zustandsnotenklasse",
    "gis_ort",
    "gis_kreis",
    "gis_bundesland",
]:
    bridge_map[c] = bridge_map[c].astype("string").str.strip()

for c in [
    "baujahr",
    "laenge",
    "breite",
    "zustandsnote",
    "geom_x",
    "geom_y",
    "traffic_dtv_latest",
    "traffic_dtv_mean",
]:
    bridge_map[c] = pd.to_numeric(
        bridge_map[c],
        errors="coerce"
    )

print("geom_x range:", bridge_map["geom_x"].min(), "to", bridge_map["geom_x"].max())
print("geom_y range:", bridge_map["geom_y"].min(), "to", bridge_map["geom_y"].max())
print("[PASS] bridge-map schema")

geom_x range: 654213.2014999986 to 1670567.606800001
geom_y range: 6005499.447400004 to 7343221.2151999995
[PASS] bridge-map schema


## 03 — Transform the canonical bridge coordinates

Controlled transformation:

```text
Source: EPSG:3857
Target: EPSG:4326 / WGS84
```

The raw `geom_x` and `geom_y` remain in the map-ready dataset for auditability.

In [4]:
transformer = Transformer.from_crs(
    "EPSG:3857",
    "EPSG:4326",
    always_xy=True,
)

x = bridge_map["geom_x"].to_numpy(dtype=float)
y = bridge_map["geom_y"].to_numpy(dtype=float)

valid_xy = (
    np.isfinite(x)
    & np.isfinite(y)
)

longitude = np.full(len(bridge_map), np.nan, dtype=float)
latitude = np.full(len(bridge_map), np.nan, dtype=float)

if valid_xy.any():
    lon_valid, lat_valid = transformer.transform(
        x[valid_xy],
        y[valid_xy],
    )
    longitude[valid_xy] = lon_valid
    latitude[valid_xy] = lat_valid

bridge_map["longitude"] = longitude
bridge_map["latitude"] = latitude

valid_geo = (
    bridge_map["longitude"].between(-180, 180)
    & bridge_map["latitude"].between(-90, 90)
)

print("Valid WGS84 coordinates:", int(valid_geo.sum()))
print("Missing/invalid coordinates:", int((~valid_geo).sum()))

if valid_geo.sum() < 50000:
    raise ValueError(
        "Too few valid WGS84 bridge coordinates after the controlled "
        "EPSG:3857 -> EPSG:4326 transformation."
    )

print("[PASS] WGS84 coordinates generated")

Valid WGS84 coordinates: 51428
Missing/invalid coordinates: 786
[PASS] WGS84 coordinates generated


## 04 — Merge current and future condition layers

The map keeps both observed and predicted current condition.

Future values are taken directly from Notebook 16.

No new prediction is performed here.

In [5]:
current_cols = [
    "bridge_id",
    "zustandsnote_observed",
    "zustandsnote_predicted",
]

missing_current = [
    c for c in current_cols
    if c not in current_df.columns
]

if missing_current:
    raise KeyError(
        f"Notebook 15 output missing: {missing_current}"
    )

future_cols = [
    "bridge_id",
    "age_years",
    "zustandsnote_t0",
    "zustandsnote_tplus_10y",
    "zustandsnote_tplus_10y_lower",
    "zustandsnote_tplus_10y_upper",
    "zustandsnote_tplus_25y",
    "zustandsnote_tplus_25y_lower",
    "zustandsnote_tplus_25y_upper",
    "zustandsnote_tplus_50y",
    "zustandsnote_tplus_50y_lower",
    "zustandsnote_tplus_50y_upper",
    "forecast_mode",
    "forecast_status",
    "extrapolation_risk_10y",
    "extrapolation_risk_25y",
    "extrapolation_risk_50y",
]

missing_future = [
    c for c in future_cols
    if c not in future_df.columns
]

if missing_future:
    raise KeyError(
        f"Notebook 16 output missing: {missing_future}"
    )

merged = (
    bridge_map
    .merge(
        current_df[current_cols],
        on="bridge_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        future_df[future_cols],
        on="bridge_id",
        how="left",
        validate="one_to_one",
    )
)

if len(merged) != 52214:
    raise ValueError(
        f"Merge changed population size: {len(merged)}"
    )

print("Merged rows:", len(merged))
print("[PASS] current + future condition layers merged")

Merged rows: 52214
[PASS] current + future condition layers merged


## 05 — Define the map-ready condition fields

The map's primary condition value depends on the selected view:

```text
CURRENT_PREDICTED
OBSERVED
PLUS_10_YEARS
PLUS_25_YEARS
PLUS_50_YEARS
```

The future fields retain their uncertainty bounds and extrapolation flags.

In [6]:
numeric_map_fields = [
    "zustandsnote_observed",
    "zustandsnote_predicted",
    "zustandsnote_t0",
    "zustandsnote_tplus_10y",
    "zustandsnote_tplus_10y_lower",
    "zustandsnote_tplus_10y_upper",
    "zustandsnote_tplus_25y",
    "zustandsnote_tplus_25y_lower",
    "zustandsnote_tplus_25y_upper",
    "zustandsnote_tplus_50y",
    "zustandsnote_tplus_50y_lower",
    "zustandsnote_tplus_50y_upper",
    "age_years",
]

for c in numeric_map_fields:
    merged[c] = pd.to_numeric(
        merged[c],
        errors="coerce"
    )

merged["map_condition_current"] = (
    merged["zustandsnote_predicted"]
)

if not np.isfinite(
    merged["map_condition_current"].dropna()
).all():
    raise ValueError(
        "Current predicted condition contains non-finite values."
    )

print(
    "Current prediction coverage:",
    int(merged["map_condition_current"].notna().sum())
)

print("[PASS] map condition fields")

Current prediction coverage: 52214
[PASS] map condition fields


## 06 — Prepare the compact map dataset

All 52,214 bridges remain in the map-ready data artifact. Bridges without valid coordinates are retained for auditability but are not rendered as map points.

This prevents the web page from carrying the full ML dataset.

In [7]:
MAP_COLUMNS = [
    "bridge_id",
    "latitude",
    "longitude",
    "bauwerksart_text",
    "baujahr",
    "age_years",
    "laenge",
    "breite",
    "baustoffklasse",
    "zustandsnote",
    "zustandsnotenklasse",
    "zustandsnote_observed",
    "zustandsnote_predicted",
    "zustandsnote_t0",
    "zustandsnote_tplus_10y",
    "zustandsnote_tplus_10y_lower",
    "zustandsnote_tplus_10y_upper",
    "zustandsnote_tplus_25y",
    "zustandsnote_tplus_25y_lower",
    "zustandsnote_tplus_25y_upper",
    "zustandsnote_tplus_50y",
    "zustandsnote_tplus_50y_lower",
    "zustandsnote_tplus_50y_upper",
    "extrapolation_risk_10y",
    "extrapolation_risk_25y",
    "extrapolation_risk_50y",
    "forecast_mode",
    "forecast_status",
    "gis_ort",
    "gis_kreis",
    "gis_bundesland",
    "traffic_dtv_latest",
    "traffic_dtv_mean",
    "geom_x",
    "geom_y",
]

# IMPORTANT:
# Do not drop bridges merely because their map coordinates are missing.
# The canonical population remains all 52,214 bridges.
# Only the browser renderer will skip rows without valid latitude/longitude.
map_ready = merged[MAP_COLUMNS].copy()

map_ready["valid_map_coordinate"] = (
    map_ready["latitude"].notna()
    & map_ready["longitude"].notna()
    & map_ready["latitude"].between(-90, 90)
    & map_ready["longitude"].between(-180, 180)
)

if len(map_ready) != 52214:
    raise ValueError(
        f"Map data population changed: expected 52,214, found {len(map_ready)}"
    )

if map_ready["bridge_id"].nunique() != 52214:
    raise ValueError(
        "Map data bridge_id is not unique across the full 52,214-bridge population."
    )

print("Map dataset rows:", len(map_ready))
print("Unique bridge IDs:", map_ready["bridge_id"].nunique())
print("Bridges with valid map coordinates:", int(map_ready["valid_map_coordinate"].sum()))
print("Bridges without valid map coordinates:", int((~map_ready["valid_map_coordinate"]).sum()))
print("[PASS] full bridge population retained")

Map dataset rows: 52214
Unique bridge IDs: 52214
Bridges with valid map coordinates: 51428
Bridges without valid map coordinates: 786
[PASS] full bridge population retained


## 07 — Export the map-ready dataset

This is the auditable data layer behind the HTML map.

In [8]:
map_ready.to_parquet(
    MAP_PARQUET,
    index=False,
)

map_ready.to_csv(
    MAP_CSV,
    index=False,
)

print("[PASS] Parquet:", MAP_PARQUET)
print("[PASS] CSV:", MAP_CSV)

[PASS] Parquet: C:\Datenanalyse\final Project\Output_PlanA-B\17_Plan_A_Germany_Web_Map\plan_a_map_data.parquet
[PASS] CSV: C:\Datenanalyse\final Project\Output_PlanA-B\17_Plan_A_Germany_Web_Map\plan_a_map_data.csv


## 08 — Build the interactive Germany map

The HTML interface provides:

- Germany-wide map;
- canvas-rendered bridge points;
- view selector;
- Bundesland filter;
- Bauwerksart filter;
- Baustoff filter;
- condition filter;
- bridge-ID search;
- bridge detail panel;
- current/future uncertainty information;
- extrapolation warnings.

The map uses Leaflet and Leaflet Canvas rendering so the 52k-point population remains usable.

In [9]:
# Compact JSON records for browser-side rendering.
# All 52,214 bridges remain in the data artifact.
# Rows without valid coordinates are retained for auditability but are not plotted.
records_df = map_ready.copy()
records_df["bridge_id"] = records_df["bridge_id"].astype(str)

records = json.loads(
    records_df.to_json(
        orient="records",
        date_format="iso",
        double_precision=6,
    )
)

records_json = json.dumps(
    records,
    ensure_ascii=False,
    separators=(",", ":"),
)

if len(records) != 52214:
    raise RuntimeError(
        f"Browser record count mismatch: expected 52,214, found {len(records)}"
    )

print("Browser records:", len(records))
print(
    "Browser-plottable records:",
    int(map_ready["valid_map_coordinate"].sum())
)
print("Embedded JSON size (MB):", round(len(records_json) / 1024**2, 2))

Browser records: 52214
Browser-plottable records: 51428
Embedded JSON size (MB): 56.35


In [10]:
html_template = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Plan A — Germany Bridge Condition Map</title>

<link
 rel="stylesheet"
 href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"
/>

<style>
html, body {
    height: 100%;
    margin: 0;
    font-family: Arial, sans-serif;
}

#map {
    height: 100%;
    width: 100%;
}

#panel {
    position: absolute;
    z-index: 1000;
    top: 12px;
    left: 12px;
    width: 330px;
    max-height: calc(100vh - 24px);
    overflow-y: auto;
    background: rgba(20, 24, 30, 0.96);
    color: #f2f2f2;
    padding: 16px;
    border-radius: 8px;
    box-shadow: 0 3px 18px rgba(0,0,0,0.35);
}

#panel h2 {
    margin: 0 0 6px 0;
    font-size: 19px;
}

#panel .subtitle {
    font-size: 12px;
    color: #c8c8c8;
    margin-bottom: 14px;
}

.control {
    margin: 9px 0;
}

.control label {
    display: block;
    font-size: 12px;
    margin-bottom: 4px;
    color: #d9d9d9;
}

select, input {
    width: 100%;
    box-sizing: border-box;
    padding: 7px;
    border-radius: 4px;
    border: 1px solid #666;
    background: #fff;
    color: #111;
}

button {
    width: 100%;
    padding: 8px;
    border: 0;
    border-radius: 4px;
    cursor: pointer;
    margin-top: 6px;
}

#stats {
    margin-top: 12px;
    padding-top: 10px;
    border-top: 1px solid #555;
    font-size: 12px;
    line-height: 1.55;
}

#detail {
    margin-top: 12px;
    padding-top: 10px;
    border-top: 1px solid #555;
    font-size: 12px;
    line-height: 1.55;
}

.warning {
    margin-top: 7px;
    padding: 7px;
    border-radius: 4px;
    background: #6b4f00;
    color: #fff;
}

.legend {
    margin-top: 12px;
    font-size: 11px;
}

.legend span {
    display: inline-block;
    width: 13px;
    height: 13px;
    margin-right: 3px;
    vertical-align: middle;
    border-radius: 50%;
}

.small {
    font-size: 10px;
    color: #aaa;
}
</style>
</head>

<body>

<div id="panel">
    <h2>Plan A — Germany Bridge Map</h2>
    <div class="subtitle">
        52,214 existing bridges · current and cohort-based future scenarios
    </div>

    <div class="control">
        <label for="view">Condition view</label>
        <select id="view">
            <option value="current">Current predicted</option>
            <option value="observed">Observed</option>
            <option value="10">+10 years</option>
            <option value="25">+25 years</option>
            <option value="50">+50 years</option>
        </select>
    </div>

    <div class="control">
        <label for="state">Bundesland</label>
        <select id="state">
            <option value="">All</option>
        </select>
    </div>

    <div class="control">
        <label for="type">Bauwerksart</label>
        <select id="type">
            <option value="">All</option>
        </select>
    </div>

    <div class="control">
        <label for="material">Baustoff</label>
        <select id="material">
            <option value="">All</option>
        </select>
    </div>

    <div class="control">
        <label for="condition">Condition range</label>
        <select id="condition">
            <option value="">All</option>
            <option value="1_2">1.0–2.0</option>
            <option value="2_3">2.0–3.0</option>
            <option value="3_4">3.0–4.0</option>
        </select>
    </div>

    <div class="control">
        <label for="bridgeSearch">Bridge ID</label>
        <input id="bridgeSearch" placeholder="Search bridge ID">
    </div>

    <button id="reset">Reset filters</button>

    <div id="stats"></div>

    <div class="legend">
        <div><span style="background:#2ca25f"></span>1–2 good</div>
        <div><span style="background:#fec44f"></span>2–3 medium</div>
        <div><span style="background:#de2d26"></span>3–4 poor</div>
    </div>

    <div id="detail">
        Select a bridge on the map.
    </div>
</div>

<div id="map"></div>

<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>

<script>
const DATA = __DATA__;

const map = L.map('map', {
    preferCanvas: true,
    zoomControl: true
}).setView([51.1657, 10.4515], 6);

L.tileLayer(
    'https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
    {
        maxZoom: 18,
        attribution: '&copy; OpenStreetMap contributors'
    }
).addTo(map);

const renderer = L.canvas({ padding: 0.5 });
let layerGroup = L.layerGroup().addTo(map);

const controls = {
    view: document.getElementById('view'),
    state: document.getElementById('state'),
    type: document.getElementById('type'),
    material: document.getElementById('material'),
    condition: document.getElementById('condition'),
    search: document.getElementById('bridgeSearch'),
    reset: document.getElementById('reset'),
    stats: document.getElementById('stats'),
    detail: document.getElementById('detail')
};

function uniqueSorted(field) {
    return [...new Set(
        DATA.map(x => x[field])
            .filter(v => v !== null && v !== undefined && String(v).trim() !== '')
            .map(v => String(v))
    )].sort((a,b) => a.localeCompare(b, 'de'));
}

function populate(select, values) {
    for (const value of values) {
        const option = document.createElement('option');
        option.value = value;
        option.textContent = value;
        select.appendChild(option);
    }
}

populate(controls.state, uniqueSorted('gis_bundesland'));
populate(controls.type, uniqueSorted('bauwerksart_text'));
populate(controls.material, uniqueSorted('baustoffklasse'));

function conditionValue(row) {
    const v = controls.view.value;

    if (v === 'current') return row.zustandsnote_predicted;
    if (v === 'observed') return row.zustandsnote_observed;
    if (v === '10') return row.zustandsnote_tplus_10y;
    if (v === '25') return row.zustandsnote_tplus_25y;
    if (v === '50') return row.zustandsnote_tplus_50y;

    return null;
}

function conditionColor(v) {
    if (v === null || v === undefined || Number.isNaN(Number(v))) {
        return '#777';
    }

    v = Number(v);

    if (v < 2.0) return '#2ca25f';
    if (v < 3.0) return '#fec44f';
    return '#de2d26';
}

function inConditionRange(v) {
    const f = controls.condition.value;

    if (!f || v === null || v === undefined || Number.isNaN(Number(v))) {
        return !f;
    }

    v = Number(v);

    if (f === '1_2') return v >= 1 && v < 2;
    if (f === '2_3') return v >= 2 && v < 3;
    if (f === '3_4') return v >= 3 && v <= 4;

    return true;
}

function filteredRows() {
    const state = controls.state.value;
    const type = controls.type.value;
    const material = controls.material.value;
    const search = controls.search.value.trim().toLowerCase();

    return DATA.filter(row => {

        if (state && String(row.gis_bundesland || '') !== state) {
            return false;
        }

        if (type && String(row.bauwerksart_text || '') !== type) {
            return false;
        }

        if (material && String(row.baustoffklasse || '') !== material) {
            return false;
        }

        if (search &&
            !String(row.bridge_id || '').toLowerCase().includes(search)) {
            return false;
        }

        const v = conditionValue(row);

        if (!inConditionRange(v)) {
            return false;
        }

        return (
            row.latitude !== null &&
            row.longitude !== null
        );
    });
}

function format(v, digits=2) {
    if (v === null || v === undefined || Number.isNaN(Number(v))) {
        return '—';
    }

    return Number(v).toLocaleString('de-DE', {
        maximumFractionDigits: digits
    });
}

function detailHtml(row) {

    const v = conditionValue(row);
    const view = controls.view.value;

    let uncertainty = '';

    if (view === '10') {
        uncertainty =
            `<br><b>95% range:</b> ${format(row.zustandsnote_tplus_10y_lower)} – ${format(row.zustandsnote_tplus_10y_upper)}`;
    }

    if (view === '25') {
        uncertainty =
            `<br><b>95% range:</b> ${format(row.zustandsnote_tplus_25y_lower)} – ${format(row.zustandsnote_tplus_25y_upper)}`;
    }

    if (view === '50') {
        uncertainty =
            `<br><b>95% range:</b> ${format(row.zustandsnote_tplus_50y_lower)} – ${format(row.zustandsnote_tplus_50y_upper)}`;
    }

    let warning = '';

    if (
        (view === '10' && row.extrapolation_risk_10y) ||
        (view === '25' && row.extrapolation_risk_25y) ||
        (view === '50' && row.extrapolation_risk_50y)
    ) {
        warning =
            `<div class="warning">Age is beyond the supported curve range for this horizon.</div>`;
    }

    return `
        <b>Bridge ID:</b> ${row.bridge_id}<br>
        <b>Bauwerksart:</b> ${row.bauwerksart_text || '—'}<br>
        <b>Baustoff:</b> ${row.baustoffklasse || '—'}<br>
        <b>Baujahr:</b> ${row.baujahr || '—'}<br>
        <b>Alter:</b> ${format(row.age_years, 0)} Jahre<br>
        <b>Länge:</b> ${format(row.laenge)} m<br>
        <b>Breite:</b> ${format(row.breite)} m<br>
        <b>Bundesland:</b> ${row.gis_bundesland || '—'}<br>
        <b>Kreis:</b> ${row.gis_kreis || '—'}<br>
        <b>Ort:</b> ${row.gis_ort || '—'}<br>
        <b>DTV latest:</b> ${format(row.traffic_dtv_latest, 0)}<br>
        <b>DTV mean:</b> ${format(row.traffic_dtv_mean, 0)}<br>
        <hr>
        <b>Observed Zustandsnote:</b> ${format(row.zustandsnote_observed)}<br>
        <b>Current predicted:</b> ${format(row.zustandsnote_predicted)}<br>
        <b>Selected view:</b> ${format(v)}
        ${uncertainty}
        ${warning}
        <div class="small">
            Future mode: ${row.forecast_mode || '—'}<br>
            Status: ${row.forecast_status || '—'}
        </div>
    `;
}

function render() {

    layerGroup.clearLayers();

    const rows = filteredRows();

    const values = rows
        .map(conditionValue)
        .filter(v => v !== null && v !== undefined && !Number.isNaN(Number(v)))
        .map(Number);

    for (const row of rows) {

        const value = conditionValue(row);

        const marker = L.circleMarker(
            [Number(row.latitude), Number(row.longitude)],
            {
                renderer: renderer,
                radius: 3,
                weight: 0.6,
                opacity: 0.75,
                fillOpacity: 0.72,
                fillColor: conditionColor(value),
                color: '#333'
            }
        );

        marker.on('click', () => {
            controls.detail.innerHTML = detailHtml(row);
        });

        marker.addTo(layerGroup);
    }

    const mean = values.length
        ? values.reduce((a,b) => a+b, 0) / values.length
        : null;

    controls.stats.innerHTML =
        `<b>Visible bridges:</b> ${rows.length.toLocaleString('de-DE')}<br>` +
        `<b>Condition mean:</b> ${format(mean)}<br>` +
        `<b>View:</b> ${controls.view.options[controls.view.selectedIndex].text}`;

    if (rows.length === 1) {
        map.setView(
            [Number(rows[0].latitude), Number(rows[0].longitude)],
            Math.max(map.getZoom(), 12)
        );
    }
}

[
    controls.view,
    controls.state,
    controls.type,
    controls.material,
    controls.condition
].forEach(el => el.addEventListener('change', render));

controls.search.addEventListener('input', render);

controls.reset.addEventListener('click', () => {
    controls.view.value = 'current';
    controls.state.value = '';
    controls.type.value = '';
    controls.material.value = '';
    controls.condition.value = '';
    controls.search.value = '';
    controls.detail.innerHTML = 'Select a bridge on the map.';
    render();
});

render();
</script>
</body>
</html>
'''

html_document = html_template.replace(
    "__DATA__",
    records_json
)

HTML_OUTPUT.write_text(
    html_document,
    encoding="utf-8"
)

print("[PASS] HTML map created:", HTML_OUTPUT)
print("HTML size (MB):", round(HTML_OUTPUT.stat().st_size / 1024**2, 2))

[PASS] HTML map created: C:\Datenanalyse\final Project\Output_PlanA-B\17_Plan_A_Germany_Web_Map\plan_a_germany_bridge_map.html
HTML size (MB): 56.46


## 09 — Create summary and manifest

The manifest records:

- exact inputs;
- input hashes;
- coordinate transformation;
- bridge population;
- valid map coordinates;
- future-condition mode;
- output artifacts.

In [11]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(chunk_size),
            b""
        ):
            h.update(chunk)
    return h.hexdigest()

summary = pd.DataFrame([
    {
        "metric": "valid_map_coordinates",
        "value": int(map_ready["valid_map_coordinate"].sum()),
    },
    {
        "metric": "missing_or_invalid_map_coordinates",
        "value": int((~map_ready["valid_map_coordinate"]).sum()),
    },
    {
        "metric": "bridge_population",
        "value": int(len(map_ready)),
    },
    {
        "metric": "valid_wgs84_coordinates",
        "value": int(
            map_ready["latitude"].notna().sum()
        ),
    },
    {
        "metric": "current_prediction_coverage",
        "value": int(
            map_ready["zustandsnote_predicted"].notna().sum()
        ),
    },
    {
        "metric": "future_10y_coverage",
        "value": int(
            map_ready["zustandsnote_tplus_10y"].notna().sum()
        ),
    },
    {
        "metric": "future_25y_coverage",
        "value": int(
            map_ready["zustandsnote_tplus_25y"].notna().sum()
        ),
    },
    {
        "metric": "future_50y_coverage",
        "value": int(
            map_ready["zustandsnote_tplus_50y"].notna().sum()
        ),
    },
    {
        "metric": "extrapolation_risk_10y",
        "value": int(
            map_ready["extrapolation_risk_10y"].fillna(False).sum()
        ),
    },
    {
        "metric": "extrapolation_risk_25y",
        "value": int(
            map_ready["extrapolation_risk_25y"].fillna(False).sum()
        ),
    },
    {
        "metric": "extrapolation_risk_50y",
        "value": int(
            map_ready["extrapolation_risk_50y"].fillna(False).sum()
        ),
    },
])

summary.to_csv(
    SUMMARY_CSV,
    index=False
)

manifest = {
    "notebook": "17_Plan_A_Germany_Web_Map",
    "population_expected": 52214,
    "population_map_ready": int(len(map_ready)),
    "input_sources": {
        "bridge_map": str(MAP_INPUT),
        "current_condition": str(CURRENT_INPUT),
        "future_condition": str(FUTURE_INPUT),
    },
    "input_sha256": {
        "bridge_map": sha256_file(MAP_INPUT),
        "current_condition": sha256_file(CURRENT_INPUT),
        "future_condition": sha256_file(FUTURE_INPUT),
    },
    "coordinate_transform": {
        "source_crs": "EPSG:3857",
        "target_crs": "EPSG:4326",
        "source_fields": ["geom_x", "geom_y"],
        "target_fields": ["longitude", "latitude"],
    },
    "future_condition": {
        "source": "Notebook 16",
        "scenario_type": "COHORT_AGE_CONDITIONED_BOOTSTRAP",
        "horizons_years": [10, 25, 50],
        "individual_longitudinal_validation": False,
    },
    "model_changed": False,
    "model_retrained": False,
    "fem_performed": False,
    "outputs": {
        "html": str(HTML_OUTPUT),
        "parquet": str(MAP_PARQUET),
        "csv": str(MAP_CSV),
        "summary": str(SUMMARY_CSV),
    },
}

MANIFEST_JSON.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("[PASS] summary:", SUMMARY_CSV)
print("[PASS] manifest:", MANIFEST_JSON)

[PASS] summary: C:\Datenanalyse\final Project\Output_PlanA-B\17_Plan_A_Germany_Web_Map\17_plan_a_map_summary.csv
[PASS] manifest: C:\Datenanalyse\final Project\Output_PlanA-B\17_Plan_A_Germany_Web_Map\17_plan_a_map_manifest.json


## 10 — Final Notebook 17 gate

The map is considered complete only when:

- 52,214 bridges are present;
- `bridge_id` is unique;
- WGS84 coordinates are valid;
- current and future condition fields exist;
- HTML map exists;
- map data Parquet/CSV exist;
- summary and manifest exist.

The output is then ready for the Plan A interface stage.

In [12]:
# Final required-field contract.
required_map_fields = [
    "bridge_id",
    "latitude",
    "longitude",
    "zustandsnote_predicted",
    "zustandsnote_tplus_10y",
    "zustandsnote_tplus_25y",
    "zustandsnote_tplus_50y",
    "gis_bundesland",
    "bauwerksart_text",
    "baustoffklasse",
]

# Notebook 16 intentionally returns NaN future values when age_years is missing.
# Therefore, the correct integrity test is:
#   every bridge with a valid age must have a future scenario;
#   bridges without age may legitimately have no scenario.
forecastable_mask = map_ready["age_years"].notna()

future_integrity = {}
for h in [10, 25, 50]:
    col = f"zustandsnote_tplus_{h}y"

    future_integrity[f"future_{h}_coverage"] = bool(
        map_ready.loc[forecastable_mask, col].notna().all()
    )

    future_integrity[f"future_{h}_expected_missing_only_without_age"] = bool(
        map_ready[col].notna().eq(forecastable_mask).all()
    )

    future_integrity[f"future_{h}_finite"] = bool(
        np.isfinite(
            map_ready.loc[map_ready[col].notna(), col].to_numpy(dtype=float)
        ).all()
    )

    future_integrity[f"future_{h}_range_1_4"] = bool(
        map_ready.loc[map_ready[col].notna(), col].between(1, 4).all()
    )

final_checks = {
    "bridge_count_52214": len(map_ready) == 52214,
    "unique_bridge_id": map_ready["bridge_id"].nunique() == 52214,

    "required_fields_present": all(
        c in map_ready.columns
        for c in required_map_fields
    ),

    # Coordinate validity is separate from population.
    "valid_latitude": bool(
        map_ready.loc[
            map_ready["valid_map_coordinate"], "latitude"
        ].between(-90, 90).all()
    ),
    "valid_longitude": bool(
        map_ready.loc[
            map_ready["valid_map_coordinate"], "longitude"
        ].between(-180, 180).all()
    ),
    "map_coordinate_flag_present": (
        "valid_map_coordinate" in map_ready.columns
    ),

    "current_condition_present": bool(
        map_ready["zustandsnote_predicted"].notna().all()
    ),

    **future_integrity,

    "html_exists": HTML_OUTPUT.exists(),
    "parquet_exists": MAP_PARQUET.exists(),
    "csv_exists": MAP_CSV.exists(),
    "summary_exists": SUMMARY_CSV.exists(),
    "manifest_exists": MANIFEST_JSON.exists(),
}

print("FINAL NOTEBOOK 17 GATE")

for name, passed in final_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

print()
print("Full bridge population:", len(map_ready))
print("Unique bridge IDs:", map_ready["bridge_id"].nunique())
print(
    "Valid map coordinates:",
    int(map_ready["valid_map_coordinate"].sum())
)
print(
    "Missing/invalid map coordinates:",
    int((~map_ready["valid_map_coordinate"]).sum())
)
print(
    "Bridges with age available:",
    int(forecastable_mask.sum())
)
print(
    "Bridges without age:",
    int((~forecastable_mask).sum())
)

for h in [10, 25, 50]:
    col = f"zustandsnote_tplus_{h}y"
    print(
        f"Future +{h}y coverage:",
        int(map_ready[col].notna().sum()),
        "/",
        int(forecastable_mask.sum()),
        "age-known bridges"
    )

if not all(final_checks.values()):
    raise RuntimeError("Notebook 17 final gate failed.")

print()
print("17 STATUS: COMPLETE")
print("Interactive map:")
print(HTML_OUTPUT)

FINAL NOTEBOOK 17 GATE
[PASS] bridge_count_52214
[PASS] unique_bridge_id
[PASS] required_fields_present
[PASS] valid_latitude
[PASS] valid_longitude
[PASS] map_coordinate_flag_present
[PASS] current_condition_present
[PASS] future_10_coverage
[PASS] future_10_expected_missing_only_without_age
[PASS] future_10_finite
[PASS] future_10_range_1_4
[PASS] future_25_coverage
[PASS] future_25_expected_missing_only_without_age
[PASS] future_25_finite
[PASS] future_25_range_1_4
[PASS] future_50_coverage
[PASS] future_50_expected_missing_only_without_age
[PASS] future_50_finite
[PASS] future_50_range_1_4
[PASS] html_exists
[PASS] parquet_exists
[PASS] csv_exists
[PASS] summary_exists
[PASS] manifest_exists

Full bridge population: 52214
Unique bridge IDs: 52214
Valid map coordinates: 51428
Missing/invalid map coordinates: 786
Bridges with age available: 52213
Bridges without age: 1
Future +10y coverage: 52213 / 52213 age-known bridges
Future +25y coverage: 52213 / 52213 age-known bridges
Future +